# nb01 Clean: Medi-Cal Managed Care Capitation Rates (v2)

**Purpose**
- Read the seven raw model CSVs saved by nb00 and standardize them for Tableau.
- Convert rate text to plain numbers, standardize plan names into a `Brand` column that joins to the enrollment data, and validate the results.
- Write one clean CSV per model to `data/clean/`.

**v2 changes, driven by the July 23 diagnostics of the real files**
- Maps the GMC file's `Rate Period` and `Calendar` headers to the canonical names.
- Maps the PACE file's `PACE Organization` to `Health Plan` and keeps its extra `AWOP` rate column.
- Fills `Health Plan` with `SCAN Health Plan` in the SCAN file (the file omits the column because SCAN is the only plan).
- Treats `$-` and bare `-` as missing instead of failed conversions.
- Forces `Model` to one spelling per file (the raw files mix `Two Plan` and `Two-Plan`, `Regional` and `Regional Model`).
- Adds a `Brand` column with standardized plan names (e.g. `LA Care`, `LA Care HP`, `LA Care Health Plan` all become `L.A. Care`) so Tableau joins to the market share project's enrollment file on Brand + Year work. Raw `Health Plan` is kept unchanged for transparency.
- The Single Plan file has no Midpoint column in the source; Midpoint stays blank there.
- New diagnostic: non-null rate values by Calendar Year, to show which years publish ranges vs a single value.
- Drops rows that are exact duplicates of another row; the source file prints a few PACE (Program of All-Inclusive Care for the Elderly) rows twice, and keeping them would double count those rates.

**What this notebook deliberately does NOT do**
- No unions and no joins. Each model stays its own file so the wildcard union, and every join to enrollment, happens inside Tableau.

**Prerequisites**
- nb00 has been run: `data/raw/` contains the seven `raw_*.csv` files.
- Run from the `tableau_capitation_rates/notebooks/` folder.

**Outputs**
- `data/clean/clean_<model>.csv` x 7. Columns: Rating Period, Calendar Year, Model, County, Health Plan, Brand, Category of Aid, Lower Bound, Midpoint, Upper Bound (+ AWOP in the PACE file only).

In [1]:
# Step 0: mirror all printed output to a text file for easy sharing
import sys
from pathlib import Path

SINK_PATH = Path.cwd() / "nb01_clean_cell_output.txt"

# Keep a handle to the notebook's original streams so re-running this cell never double-wraps
_orig_out = getattr(sys, "_nb_orig_stdout", sys.stdout)
_orig_err = getattr(sys, "_nb_orig_stderr", sys.stderr)
sys._nb_orig_stdout, sys._nb_orig_stderr = _orig_out, _orig_err

class _Tee:
    def __init__(self, stream, fh):
        self.stream, self.fh = stream, fh
    def write(self, data):
        self.stream.write(data)
        self.fh.write(data)
        self.fh.flush()
    def flush(self):
        self.stream.flush()
        self.fh.flush()

_sink = open(SINK_PATH, "w")   # a fresh run of this cell starts the file over
sys.stdout = _Tee(_orig_out, _sink)
sys.stderr = _Tee(_orig_err, _sink)
print(f"Mirroring cell output to {SINK_PATH.name} (attach this file in the chat)")

Mirroring cell output to nb01_clean_cell_output.txt (attach this file in the chat)
All 7 raw files found in /Users/trinidadcisneros/Documents/Development/trinidadcisneros.github.io/folders/ds_blogs/projects/tableau/tableau_capitation_rates/data/raw
Output columns: ['Rating Period', 'Calendar Year', 'Model', 'County', 'Health Plan', 'Brand', 'Category of Aid', 'Lower Bound', 'Midpoint', 'Upper Bound']
Helpers ready; 40 plan name variants mapped

=== raw_two_plan.csv -> clean_two_plan.csv ===
  rows in 4009, rows out 4009

=== raw_cohs.csv -> clean_cohs.csv ===
  dropped 6 rows with no rate values at all
  rows in 4484, rows out 4478

=== raw_gmc.csv -> clean_gmc.csv ===
  rows in 1045, rows out 1045

=== raw_regional.csv -> clean_regional.csv ===
  rows in 2393, rows out 2393

=== raw_single_plan.csv -> clean_single_plan.csv ===
  note: the Single Plan source publishes no Midpoint; column left blank
  rows in 414, rows out 414

=== raw_scan.csv -> clean_scan.csv ===
  note: Health Plan 

In [2]:
# Step 1: imports, paths, and input check
from pathlib import Path
import re
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILES = {
    "raw_two_plan.csv":    ("clean_two_plan.csv",    "Two-Plan"),
    "raw_cohs.csv":        ("clean_cohs.csv",        "COHS"),
    "raw_gmc.csv":         ("clean_gmc.csv",         "GMC"),
    "raw_regional.csv":    ("clean_regional.csv",    "Regional"),
    "raw_single_plan.csv": ("clean_single_plan.csv", "Single Plan"),
    "raw_scan.csv":        ("clean_scan.csv",        "SCAN"),
    "raw_pace.csv":        ("clean_pace.csv",        "PACE"),
}

missing = [f for f in MODEL_FILES if not (RAW_DIR / f).exists()]
assert not missing, f"Missing raw files (run nb00 first): {missing}"
print("All 7 raw files found in", RAW_DIR)

In [3]:
# Step 2: column standardization
CANONICAL = ["Rating Period", "Calendar Year", "Model", "County",
             "Health Plan", "Category of Aid", "Lower Bound", "Midpoint", "Upper Bound"]
OUTPUT_COLS = CANONICAL[:5] + ["Brand"] + CANONICAL[5:]   # Brand sits after Health Plan

def norm(name):
    """Normalize a name for matching: lowercase, alphanumeric only."""
    return re.sub(r"[^a-z0-9]", "", str(name).lower())

NORM_TO_CANONICAL = {norm(c): c for c in CANONICAL}
NORM_TO_CANONICAL.update({
    "rateperiod": "Rating Period",       # GMC
    "ratingperioddates": "Rating Period",
    "calendar": "Calendar Year",         # GMC
    "year": "Calendar Year",
    "plan": "Health Plan",
    "healthplanname": "Health Plan",     # Single Plan
    "planname": "Health Plan",
    "paceorganization": "Health Plan",   # PACE
    "aidcategory": "Category of Aid",
    "categoriesofaid": "Category of Aid",
    "lower": "Lower Bound",
    "upper": "Upper Bound",
    "lowerboundrate": "Lower Bound",
    "midpointrate": "Midpoint",
    "upperboundrate": "Upper Bound",
})
EXTRA_NUMERIC = {"raw_pace.csv": ["AWOP"]}   # PACE-only extra rate column, kept and cleaned

def standardize_columns(df, file_name):
    rename, unmapped = {}, []
    for c in df.columns:
        target = NORM_TO_CANONICAL.get(norm(c))
        if target:
            rename[c] = target
        elif norm(c) != "awop":
            unmapped.append(c)
    df = df.rename(columns=rename)
    df = df.rename(columns={c: "AWOP" for c in df.columns if norm(c) == "awop"})
    if unmapped:
        print(f"  WARNING {file_name}: unmapped columns kept as-is: {unmapped}")
    return df

print("Output columns:", OUTPUT_COLS)

In [4]:
# Step 3: value cleaning and brand standardization helpers
def clean_text(s):
    """Trim and collapse internal whitespace in a text Series."""
    return s.astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

def to_number(series, col_label, file_name):
    """Convert rate text like '$1,234.56' to float; '$-', '-', blank become missing."""
    s = series.astype("string").str.strip()
    stripped = s.str.replace(r"[$,\s]", "", regex=True)
    stripped = stripped.str.replace(r"^\((.*)\)$", r"-\1", regex=True)
    stripped = stripped.mask(stripped.str.lower().isin(
        {"", "-", "--", "na", "n/a", "none", "null", "tbd"}))
    out = pd.to_numeric(stripped, errors="coerce")
    failed = stripped.notna() & out.isna()
    if failed.any():
        bad = sorted(s[failed].unique().tolist())[:10]
        print(f"  FAILED CONVERSION {file_name} / {col_label}: {failed.sum()} values, e.g. {bad}")
    return out

def to_year(series, file_name):
    s = series.astype("string").str.extract(r"(\d{4})", expand=False)
    out = pd.to_numeric(s, errors="coerce").astype("Int64")
    if out.isna().any():
        print(f"  WARNING {file_name}: {out.isna().sum()} rows with no parseable Calendar Year")
    return out

# Standardized brand per raw plan name (matched on the normalized form).
# 'L.A. Care' and 'Health Net' must match the Brand values in
# tableau_la_market_share/data/la_market_share_clean.csv for the Tableau join.
BRAND_MAP = {
    "lacare": "L.A. Care", "lacarehp": "L.A. Care", "lacarehealthplan": "L.A. Care",
    "healthnetofcalifornia": "Health Net", "healthnet": "Health Net",
    "anthembluecross": "Anthem Blue Cross",
    "calvivahealth": "CalViva Health",
    "contracostahp": "Contra Costa Health Plan", "contracostahealthplan": "Contra Costa Health Plan",
    "hpofsanjoaquin": "Health Plan of San Joaquin", "healthplanofsanjoaquin": "Health Plan of San Joaquin",
    "inlandempirehp": "Inland Empire Health Plan", "inlandempirehealthplan": "Inland Empire Health Plan",
    "kaiserfoundationhp": "Kaiser Permanente", "kaiserfoundationhealthplan": "Kaiser Permanente",
    "kaiserpermanente": "Kaiser Permanente",
    "kernhealthsystems": "Kern Health Systems",
    "molinahealthcare": "Molina Healthcare",
    "sanfranciscohealthplan": "San Francisco Health Plan",
    "santaclarafamilyhp": "Santa Clara Family Health Plan",
    "santaclarafamilyhealthplan": "Santa Clara Family Health Plan",
    "alamedaallianceforhealth": "Alameda Alliance for Health",
    "caloptima": "CalOptima",
    "cencalhealth": "CenCal Health",
    "centralcaallianceforhealth": "Central California Alliance for Health",
    "centralcaliforniaallianceforhealth": "Central California Alliance for Health",
    "goldcoast": "Gold Coast Health Plan", "goldcoasthealthplan": "Gold Coast Health Plan",
    "healthplanofsanmateo": "Health Plan of San Mateo",
    "partnershiphealthplan": "Partnership Health Plan",
    "partnershiphealthplanofcalifornia": "Partnership Health Plan",
    "aetnabetterhealth": "Aetna Better Health",
    "blueshieldofcalifornia": "Blue Shield Promise",
    "blueshieldpromisehealthplanofcalifornia": "Blue Shield Promise",
    "communityhealthgroup": "Community Health Group",
    "unitedhealthcare": "UnitedHealthcare",
    "cahealthwellness": "California Health and Wellness",
    "californiahealthandwellness": "California Health and Wellness",
    "communityhealthplanimperialvalley": "Community Health Plan Imperial Valley",
    "scanhealthplan": "SCAN Health Plan",
}

def add_brand(df, file_name):
    """Map Health Plan to a standardized Brand; report names the map does not know."""
    keys = df["Health Plan"].map(norm)
    df["Brand"] = keys.map(BRAND_MAP)
    unknown = df.loc[df["Brand"].isna() & df["Health Plan"].notna(), "Health Plan"].unique().tolist()
    if unknown:
        # PACE organizations are expected here: dozens of local PACE programs, kept as-is
        label = "note (PACE orgs keep their own name)" if file_name == "raw_pace.csv" else "WARNING unmapped plan names"
        print(f"  {label} {file_name}: {sorted(unknown)[:12]}{' ...' if len(unknown) > 12 else ''}")
    df["Brand"] = df["Brand"].fillna(df["Health Plan"])
    return df

print("Helpers ready;", len(BRAND_MAP), "plan name variants mapped")

In [5]:
# Step 4: clean every model file
clean_frames = {}
for raw_name, (clean_name, model_label) in MODEL_FILES.items():
    print(f"\n=== {raw_name} -> {clean_name} ===")
    df = pd.read_csv(RAW_DIR / raw_name, dtype=str)
    n_in = len(df)
    df = standardize_columns(df, raw_name)

    for col in CANONICAL:
        if col not in df.columns:
            df[col] = pd.NA
            if raw_name == "raw_scan.csv" and col == "Health Plan":
                df[col] = "SCAN Health Plan"
                print("  note: Health Plan absent in SCAN file, filled with 'SCAN Health Plan'")
            elif raw_name == "raw_single_plan.csv" and col == "Midpoint":
                print("  note: the Single Plan source publishes no Midpoint; column left blank")
            else:
                print(f"  WARNING {raw_name}: column '{col}' not found, created empty. "
                      f"Actual columns were: {list(df.columns)}")

    # Text columns; Model forced to one spelling per file; Rating Period dashes normalized
    for col in ["Rating Period", "County", "Health Plan", "Category of Aid"]:
        df[col] = clean_text(df[col])
    df["Model"] = model_label
    df["Rating Period"] = df["Rating Period"].str.replace(r"\s*-\s*", "-", regex=True)

    # Numeric columns
    df["Calendar Year"] = to_year(df["Calendar Year"], raw_name)
    rate_cols = ["Lower Bound", "Midpoint", "Upper Bound"] + EXTRA_NUMERIC.get(raw_name, [])
    for col in rate_cols:
        if col in df.columns:
            df[col] = to_number(df[col], col, raw_name)

    # Standardized Brand for Tableau joins
    df = add_brand(df, raw_name)

    # Drop rows that are exact duplicates of another row (every column identical).
    # The DHCS source prints a few rows twice (e.g. Central Valley PACE, late 2021);
    # keeping them would double count those rates in any sum or average.
    before = len(df)
    df = df.drop_duplicates()
    exact_dups = before - len(df)
    if exact_dups:
        print(f"  dropped {exact_dups} exact duplicate rows (identical in every column)")

    # Drop rows with no rate information at all
    before = len(df)
    df = df[~(df["Lower Bound"].isna() & df["Midpoint"].isna() & df["Upper Bound"].isna())].copy()
    dropped = before - len(df)
    if dropped:
        print(f"  dropped {dropped} rows with no rate values at all")

    df = df[OUTPUT_COLS + [c for c in EXTRA_NUMERIC.get(raw_name, []) if c in df.columns]]
    clean_frames[clean_name] = df
    print(f"  rows in {n_in}, rows out {len(df)}")

In [6]:
# Step 5: validation checks
print("Validation\n" + "=" * 40)
for name, df in clean_frames.items():
    issues = []

    yrs = df["Calendar Year"].dropna()
    if len(yrs) and (yrs.min() < 2021 or yrs.max() > 2026):
        issues.append(f"years outside 2021-2026: {sorted(yrs.unique().tolist())}")

    ordered = df.dropna(subset=["Lower Bound", "Midpoint", "Upper Bound"])
    bad_order = ordered[(ordered["Lower Bound"] > ordered["Midpoint"]) |
                        (ordered["Midpoint"] > ordered["Upper Bound"])]
    if len(bad_order):
        issues.append(f"{len(bad_order)} rows violate Lower <= Midpoint <= Upper")

    key = ["Model", "County", "Health Plan", "Category of Aid", "Calendar Year", "Rating Period"]
    dups = df.duplicated(subset=key, keep=False).sum()
    if dups:
        issues.append(f"{dups} rows share a duplicate key {key}")

    status = "OK" if not issues else "CHECK"
    nan_rates = int(df[["Lower Bound", "Midpoint", "Upper Bound"]].isna().sum().sum())
    print(f"{name:24s} {status}  rows {len(df):>5}  NaN rate cells {nan_rates}")
    for i in issues:
        print(f"    - {i}")

In [7]:
# Step 5b: which years publish ranges vs a single value (explains the NaN pattern)
print("Non-null rate values by Calendar Year (rows with Lower / Midpoint / Upper)\n")
for name, df in clean_frames.items():
    g = df.groupby("Calendar Year", dropna=False).agg(
        rows=("Model", "size"),
        lower=("Lower Bound", "count"),
        midpoint=("Midpoint", "count"),
        upper=("Upper Bound", "count"))
    print(f"=== {name} ===")
    print(g.to_string(), "\n")

In [8]:
# Step 5c: Los Angeles spot check with Brand values
tp = clean_frames["clean_two_plan.csv"]
la = tp[tp["County"].str.contains("Los Angeles", case=False, na=False)]
print(f"Los Angeles rows in clean Two-Plan file: {len(la)}")
assert len(la) > 0, "No Los Angeles rows found in the Two-Plan file; inspect raw data"
print("LA brands by year:")
print(la.groupby(["Calendar Year", "Brand"]).size().to_string())
print()
print(la.head(12).to_string(index=False))

In [9]:
# Step 6: write the clean CSVs
for name, df in clean_frames.items():
    out = CLEAN_DIR / name
    df.to_csv(out, index=False)
    print(f"saved {name:24s} {len(df):>5} rows  {out.stat().st_size:>9,} bytes")

print("\nREADY FOR TABLEAU")
print("Union: connect to any one clean_*.csv, then wildcard union on clean_*.csv in", CLEAN_DIR)

In [10]:
# Final step: confirm the output sink
sys.stdout.flush()
print(f"\nAll printed output saved to: {SINK_PATH}")
print(f"File size: {SINK_PATH.stat().st_size:,} bytes")

**Next step**
- Send `nb01_clean_cell_output.txt` back to the chat, especially any WARNING or FAILED CONVERSION lines and the Step 5b year tables.
- The enrollment file for the Tableau join needs no new prep: Tableau connects directly to `tableau_la_market_share/data/la_market_share_clean.csv` and joins on Brand + Year.
- After the outputs check out, the Tableau build starts: wildcard union of the seven clean files, one step at a time.
- If a cell errors with a red traceback, copy that traceback separately, since the mirror captures printed output only.